# NumGLUE A3 — generate `cara` (Kaggle, vLLM)

Jalankan di **Kaggle GPU T4 x2**. Tanpa batas token (model lokal).

Input  : `numglue_{split}_id.jsonl` (hasil A2 translate, upload via **+ Add Data**)
Output : `/kaggle/working/numglue_{split}_final.jsonl`  {soal, cara, jawaban, source, cara_source}

```
A3a generate N kandidat (blind)  -->  A3b cek benar via math_verify (gratis, jawaban numerik)
   -->  A3c: ada yang benar? pakai itu (verified). Tidak? hinted (model sama diberi gold). 100% coverage.
```
**1 model saja** (`DeepSeek-R1-Distill-Qwen-7B`). Checkpoint per chunk → **resumable**.


## 0. Setup

In [ ]:
!pip install -q vllm math-verify
# Hapus flashinfer: di T4 JIT-nya gagal link -lcuda. Tanpa flashinfer, vLLM fallback ke xformers/Triton.
!pip uninstall -y flashinfer flashinfer-python flashinfer-jit-cache 2>/dev/null || true


In [ ]:
import os
# Kaggle T4: libcuda.so tak ada di lib64 -> flashinfer JIT butuh ini. Symlink dari stubs.
stub='/usr/local/cuda/lib64/stubs/libcuda.so'; tgt='/usr/local/cuda/lib64/libcuda.so'
if os.path.exists(stub) and not os.path.exists(tgt):
    os.symlink(stub, tgt); print('symlinked libcuda.so')
os.environ['VLLM_USE_FLASHINFER_SAMPLER']='0'   # hindari JIT -lcuda
os.environ['VLLM_ATTENTION_BACKEND']='XFORMERS'   # T4: hindari flashinfer attention (gagal link -lcuda)
print('flashinfer sampler off')


## 1. Konfigurasi + cari input

In [ ]:
import glob, json
from pathlib import Path

SPLITS    = ['test', 'dev', 'train']   # mulai ['test'] dulu
MODEL     = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'
N_CAND    = 4        # kandidat blind per soal (A3a)
MAX_TOKENS= 3072
CHUNK     = 200      # checkpoint tiap CHUNK soal
TP_SIZE   = 2        # Kaggle T4 x2
OUTDIR    = Path('/kaggle/working')

def find_input(split):
    hits = glob.glob(f'/kaggle/input/**/numglue_{split}_id.jsonl', recursive=True)
    return hits[0] if hits else None

INPUTS = {s: find_input(s) for s in SPLITS}
for s, p in INPUTS.items(): print(f'{s:5} input: {p}')


## 2. Helper (inline)

In [ ]:
import re

def read_jsonl(p): return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]

def strip_think(t):
    i = t.find('</think>'); return t[i+8:].strip() if i>=0 else t.strip()

def extract_boxed(text):
    if not text: return None
    s = text.rfind(r'\boxed')
    if s == -1: return None
    i = s + 6
    while i < len(text) and text[i] != '{': i += 1
    if i >= len(text): return None
    depth = 0; out = []
    for ch in text[i:]:
        if ch == '{':
            depth += 1
            if depth == 1: continue
        elif ch == '}':
            depth -= 1
            if depth == 0: return ''.join(out).strip()
        out.append(ch)
    return None

def _norm(s):
    s = str(s).strip(); b = extract_boxed(s)
    if b is not None: s = b
    s = s.strip().strip('$').strip()
    for r in [r'\!', r'\,', r'\;', r'\left', r'\right']: s = s.replace(r, '')
    s = s.replace(r'\dfrac', r'\frac').replace(r'\tfrac', r'\frac').replace('{,}', '.').rstrip('.')
    s = re.sub(r'\s+', '', s)
    if re.fullmatch(r'-?\d+,\d+', s): s = s.replace(',', '.')
    return s.lower()

def answers_equivalent(pred, gold):
    if not gold: return False
    if _norm(pred) and _norm(pred) == _norm(gold): return True
    try:
        from math_verify import parse, verify
        g = parse(gold if ('\\boxed' in gold or '$' in gold) else f'${gold}$')
        p = parse(pred if ('\\boxed' in pred or '$' in pred) else f'${pred}$')
        return bool(verify(g, p))
    except Exception:
        return False

PROMPT_COT = ('Selesaikan soal matematika berikut. Tunjukkan langkah-langkah penyelesaian secara '
  'rinci dan sistematis dalam Bahasa Indonesia. Pastikan jawaban akhir (dan hanya jawaban) berada '
  r'di dalam \boxed{{}}.' '\n\n{soal}')
PROMPT_HINT = ('Selesaikan soal matematika berikut. Jawaban akhir yang BENAR sudah diketahui: {jawaban}. '
  'Tuliskan langkah-langkah penyelesaian yang rinci dan logis dalam Bahasa Indonesia sehingga sampai '
  'pada jawaban tersebut. Jangan hanya menuliskan jawaban; tunjukkan penalarannya. Akhiri dengan '
  r'jawaban akhir di dalam \boxed{{}}.' '\n\nSoal:\n{soal}')
print('helpers siap')


## 3. Muat model (vLLM, sekali)

In [ ]:
from vllm import LLM, SamplingParams
llm = LLM(model=MODEL, dtype='float16', gpu_memory_utilization=0.90,
          max_model_len=max(4096, MAX_TOKENS+1024), trust_remote_code=True,
          tensor_parallel_size=TP_SIZE)
print('model loaded')


## 4. Generate + correctness + assemble (per split, checkpoint per chunk)
Pass-1 blind N kandidat → cek benar (math). Pass-2 hinted utk yang belum kepecah.


In [ ]:
sp_cot  = SamplingParams(n=N_CAND, temperature=0.7, top_p=0.95, max_tokens=MAX_TOKENS)
sp_hint = SamplingParams(n=1, temperature=0.3, top_p=0.95, max_tokens=MAX_TOKENS)

for split in SPLITS:
    inp = INPUTS[split]
    if not inp: print(f'SKIP {split}: input tak ada'); continue
    out = OUTDIR / f'numglue_{split}_final.jsonl'
    problems = read_jsonl(inp)
    done = {(json.loads(l)['soal']) for l in open(out, encoding='utf-8')} if out.exists() else set()
    todo = [pr for pr in problems if pr['soal'] not in done]
    print(f'\n== {split} == total {len(problems)}, sisa {len(todo)}')

    with open(out, 'a', encoding='utf-8') as f:
        for start in range(0, len(todo), CHUNK):
            chunk = todo[start:start+CHUNK]
            # Pass-1 blind
            convos = [[{'role':'user','content':PROMPT_COT.format(soal=pr['soal'])}] for pr in chunk]
            outs = llm.chat(convos, sp_cot)
            res = {}; unsolved = []
            for pr, o in zip(chunk, outs):
                gold = str(pr.get('jawaban','')).strip()
                good = [strip_think(c.text) for c in o.outputs
                        if (extract_boxed(c.text) is not None and answers_equivalent(extract_boxed(c.text), gold))]
                if good: res[pr['soal']] = (min(good, key=len), 'verified')
                else: unsolved.append(pr)
            # Pass-2 hinted
            if unsolved:
                convos2 = [[{'role':'user','content':PROMPT_HINT.format(soal=pr['soal'], jawaban=pr.get('jawaban',''))}] for pr in unsolved]
                outs2 = llm.chat(convos2, sp_hint)
                for pr, o in zip(unsolved, outs2):
                    res[pr['soal']] = (strip_think(o.outputs[0].text), 'hinted')
            for pr in chunk:
                cara, csrc = res[pr['soal']]
                f.write(json.dumps({'soal':pr['soal'],'cara':cara,'jawaban':pr.get('jawaban',''),
                    'source':pr.get('source','numglue'),'cara_source':csrc}, ensure_ascii=False)+'\n')
            f.flush()
            print(f'   checkpoint {min(start+CHUNK,len(todo))}/{len(todo)}')


## 5. Cek hasil

In [ ]:
from collections import Counter
for split in SPLITS:
    out = OUTDIR / f'numglue_{split}_final.jsonl'
    if not out.exists(): print(f'{split}: belum ada'); continue
    rows = read_jsonl(out); cs = Counter(r['cara_source'] for r in rows)
    print(f'{split:5} n={len(rows):5} cara_source={dict(cs)}')
print('\nDownload numglue_{split}_final.jsonl dari Output panel.')
